# 02 — Gold Feature Engineering

**Input :** `hive_metastore.gold.gold_dataset`  
**Output:** `hive_metastore.gold.gold_features`

### Label design — look-ahead binary

Following the framing of Ahmadi et al. (2024, *Smart Grids and Sustainable Energy*)  
who predicted distribution-transformer overload within 1–3 hour horizons, we define  
two binary labels based on a **forward-looking** window (no data leakage):

| Column | Definition |
|---|---|
| `label_4h`  | 1 if `max(load_ratio_c, load_ratio_v) > 1` in any of the next **16 steps** (4 h)  |
| `label_24h` | 1 if `max(load_ratio_c, load_ratio_v) > 1` in any of the next **96 steps** (24 h) |

Both horizons are evaluated to quantify the precision–recall trade-off that comes  
with extending lead time (Pinci et al., 2024; ScienceDirect).

### Train / Test split

Data range: **2023-07-04 → 2024-07-03** (~12 months).  
Chronological 80/20 split — no shuffling, no validation set (handled via `early_stopping_rounds` inside model training).

| Split | Period | Rationale |
|---|---|---|
| **train** | 2023-07-04 → 2024-03-31 | ~9 months; covers both winter load and early summer |
| **test**  | 2024-04-01 → 2024-07-03 | ~3 months; spring→summer transition — hardest operational period |

A **purge gap** of 24 h (96 rows @ 15-min cadence) is applied before the test boundary.  
Rows in the purge zone receive `null` labels and are dropped — their look-ahead windows  
would otherwise cross into the test partition and cause label leakage.

### Class imbalance — model-level only

No resampling is applied to the dataset. SMOTE and random oversampling break temporal  
ordering and invalidate window-based features. Imbalance is addressed at the model level:

| Model | Parameter |
|---|---|
| XGBoost | `scale_pos_weight = neg/pos` |
| Logistic Regression | `weightCol` (positive rows weighted by `neg/pos`) |
| LSTM | `pos_weight` in `BCEWithLogitsLoss` |

The exact per-label, per-split ratios are computed and stored in `scaler.json`.

### Centralised scaling

Scaler statistics are **fit on training rows only** and applied to all rows.  
Parameters are persisted to `dbfs:/gold_features_meta/scaler.json` for inference parity.

| Column group | Treatment |
|---|---|
| Continuous numerics | z-score |
| Heavy-tailed counts / max | log1p → z-score |
| Cyclical encodings (sin/cos) | keep raw — bounded [-1, 1] |
| Binary flags | keep raw |
| Categoricals (ID_prefix, CONCELHO) | keep raw — indexed by model pipelines |

### Engineered features

| Group | Columns |
|---|---|
| Load ratio | `load_ratio_c`, `load_ratio_v` |
| Rolling stats | `{current,voltage}_{mean,std,max}_{1h,1d,7d}` (look-back only) |
| Lag features | `{current,voltage}_lag_{15m,1h,1d}` |
| Weather derived | `temp_mean_1d`, `temp_mean_7d`, `precip_sum_1d` |
| Temporal | `hour`, `day_of_week`, `month`, `is_weekend` |

> **Leakage note:** all rolling and lag features use `rowsBetween(-n, -1)` —  
> the current row is excluded. Labels use `rowsBetween(1, H)` — strictly future rows.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
import json
import math
import builtins
from pyspark.sql import functions as F
from pyspark.sql import Window

ID  = "ID_prefix"
TS  = "DATE"

SRC = "hive_metastore.gold.gold_dataset"
DST = "hive_metastore.gold.gold_features"
SCALER_PATH = "dbfs:/gold_features_meta/scaler.json"

# ── Cadence & horizons ─────────────────────────────────────────────────────────
CADENCE_MIN = 15
STEPS_PER_H = 60 // CADENCE_MIN   # 4

H_4H  = 16   # 4 hours  = 16 steps
H_24H = 96   # 24 hours = 96 steps

SIGNAL_COLS = ["current", "voltage"]

ROLLING_WINDOWS = {
    "1h":  4,    # 1 hour
    "1d":  96,   # 1 day
    "7d":  672,  # 7 days
}

LAG_STEPS = {
    "15m": 1,
    "1h":  4,
    "1d":  96,
}

# ── Train / test split ─────────────────────────────────────────────────────────
# Chronological 80/20 — data runs 2023-07-04 to 2024-07-03
# Test starts 2024-04-01: spring-to-summer transition = hardest evaluation period
TRAIN_END  = "2024-04-01"   # exclusive upper bound for train
TEST_START = "2024-04-01"   # inclusive lower bound for test

# ── Purge gap ──────────────────────────────────────────────────────────────────
# Drop rows whose look-ahead window (max horizon = 24 h) crosses the split boundary.
# This prevents the label of a training row from being computed using test-period data.
MAX_HORIZON_H = 24
PURGE_STEPS   = MAX_HORIZON_H * STEPS_PER_H   # 96 rows = 24 h @ 15-min cadence
PURGE_SEC     = PURGE_STEPS * CADENCE_MIN * 60

print(f"Split  : train < {TRAIN_END}  |  test >= {TEST_START}")
print(f"Purge  : {PURGE_STEPS} rows ({MAX_HORIZON_H} h) before {TRAIN_END}")

## 1 · Load gold_dataset

In [0]:
df = spark.read.table(SRC)
print(f"Loaded {df.count():,} rows  |  {len(df.columns)} columns")
df.printSchema()

## 2 · Load ratio (normalised loading)

`load_ratio_c = current / H_LIM_C`  
`load_ratio_v = voltage / H_LIM_V`

Values > 1.0 indicate the transformer is above its alarm threshold.  
These are the most informative single features and the basis for both labels.

In [0]:
df = (
    df
    .withColumn(
        "load_ratio_c",
        F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(F.lit(None))
    )
    .withColumn(
        "load_ratio_v",
        F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(F.lit(None))
    )
)

display(
    df.select(ID, TS, "current", "H_LIM_C", "load_ratio_c",
                      "voltage", "H_LIM_V", "load_ratio_v").limit(10)
)

## 3 · Look-ahead binary labels

For each row at time **t**, the label = 1 if the transformer will breach its  
alarm threshold (load_ratio > 1 on current **or** voltage) at **any** point  
in the look-ahead window `[t+1, t+H]`.

The forward window uses `rowsBetween(1, H)` — strictly after the current row —  
so no information from the future leaks into the features.

```
label_4h  : H = 16 steps (4 h)   — short horizon, high precision
label_24h : H = 96 steps (24 h)  — day-ahead,    lower precision
```

In [0]:
# Instantaneous overload flag (1 when either signal is above its limit)
df = df.withColumn(
    "_is_overloaded",
    (
        (F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1)
    ).cast("int")
)

In [0]:
def make_lookahead_label(df, horizon_steps, out_col):
    """
    Binary label: 1 if the transformer will be overloaded
    in any of the next `horizon_steps` rows (strictly future — no leakage).
    """
    w_future = (
        Window
        .partitionBy(ID)
        .orderBy(F.col(TS).cast("long"))
        .rowsBetween(1, horizon_steps)   # [t+1 … t+H], current row excluded
    )
    return df.withColumn(
        out_col,
        F.max("_is_overloaded").over(w_future).cast("int")
    )


df = make_lookahead_label(df, H_4H,  "label_4h")
df = make_lookahead_label(df, H_24H, "label_24h")

# Drop the helper column
df = df.drop("_is_overloaded")

print("Label distribution — 4h horizon:")
df.groupBy("label_4h").count().orderBy("label_4h").show()

print("Label distribution — 24h horizon:")
df.groupBy("label_24h").count().orderBy("label_24h").show()

In [0]:
# Rows at the tail of each transformer's history have no future window → label is null.
# Drop these rows — they cannot be used for supervised training.
before = df.count()
df = df.filter(F.col("label_4h").isNotNull() & F.col("label_24h").isNotNull())
after  = df.count()
print(f"Dropped {before - after:,} tail rows with null labels  |  {after:,} rows remaining")

## 4 · Rolling statistics (look-back, no leakage)

| Window name | Steps | Duration |
|---|---|---|
| 1h | 4 | 1 hour |
| 1d | 96 | 1 day |
| 7d | 672 | 7 days |

Each window uses `rowsBetween(-n, -1)` — strictly before the current row.

In [0]:
for w_name, n_rows in ROLLING_WINDOWS.items():
    w_back = (
        Window
        .partitionBy(ID)
        .orderBy(F.col(TS).cast("long"))
        .rowsBetween(-n_rows, -1)   # look-back only
    )
    for col in SIGNAL_COLS:
        df = (
            df
            .withColumn(f"{col}_mean_{w_name}", F.mean(col).over(w_back))
            .withColumn(f"{col}_std_{w_name}",  F.stddev(col).over(w_back))
            .withColumn(f"{col}_max_{w_name}",  F.max(col).over(w_back))
        )

print(f"Rolling features added ✅  — {len(df.columns)} total columns")
roll_cols = [c for c in df.columns if any(w in c for w in ROLLING_WINDOWS)]
display(df.select([ID, TS] + roll_cols[:8]).limit(5))

## 5 · Lag features

| Lag name | Steps | Lead time |
|---|---|---|
| 15m | 1 | Previous reading |
| 1h  | 4 | 1 hour ago |
| 1d  | 96 | 24 hours ago |

In [0]:
w_order = Window.partitionBy(ID).orderBy(TS)

for lag_name, n in LAG_STEPS.items():
    for col in SIGNAL_COLS:
        df = df.withColumn(f"{col}_lag_{lag_name}", F.lag(col, n).over(w_order))

print(f"Lag features added ✅  — {len(df.columns)} total columns")
lag_cols = [c for c in df.columns if "_lag_" in c]
display(df.select([ID, TS] + lag_cols).limit(5))

## 6 · Weather derived features

Raw hourly weather values are kept as-is. Three derived features are added on top:

| Feature | Window | Rationale |
|---|---|---|
| `temp_mean_1d` | 24 h look-back | Captures same-day thermal load on the transformer |
| `temp_mean_7d` | 7 d look-back | Captures sustained hot-spell effect |
| `precip_sum_1d` | 24 h look-back | Cumulative precipitation proxies storm episodes |

In [0]:
TEMP_COL   = "temperatura_media_do_ar_horaria_c"
PRECIP_COL = "precipitacao_horaria_mm"

w_1d = Window.partitionBy(ID).orderBy(F.col(TS).cast("long")).rowsBetween(-96, -1)
w_7d = Window.partitionBy(ID).orderBy(F.col(TS).cast("long")).rowsBetween(-672, -1)

df = (
    df
    .withColumn("temp_mean_1d",  F.mean(TEMP_COL).over(w_1d))
    .withColumn("temp_mean_7d",  F.mean(TEMP_COL).over(w_7d))
    .withColumn("precip_sum_1d", F.sum(PRECIP_COL).over(w_1d))
)

display(df.select(ID, TS, TEMP_COL, "temp_mean_1d", "temp_mean_7d", PRECIP_COL, "precip_sum_1d").limit(5))

## 7 · Temporal features

In [0]:
df = (
    df
    .withColumn("hour",        F.hour(TS))
    .withColumn("day_of_week", F.dayofweek(TS))   # 1 = Sunday … 7 = Saturday
    .withColumn("month",       F.month(TS))
    .withColumn("is_weekend",  F.dayofweek(TS).isin([1, 7]).cast("int"))
)

display(df.select(ID, TS, "hour", "day_of_week", "month", "is_weekend").limit(5))

## 8 · Train / test split + purge gap

The purge gap removes the last 24 h of training rows (96 rows @ 15-min cadence)  
whose look-ahead windows would cross into the test period, preventing label leakage.

In [0]:
# Compute Unix timestamp of the split boundary
train_end_ts = spark.sql(f"SELECT unix_timestamp(to_timestamp('{TRAIN_END}')) AS ts").first()["ts"]
purge_start_ts = train_end_ts - PURGE_SEC

# Flag rows in the purge zone (last 24 h before test boundary)
in_purge_zone = (
    (F.col(TS).cast("long") >= purge_start_ts) &
    (F.col(TS).cast("long") <  train_end_ts)
)

# Null out labels for purge-zone rows so they get dropped cleanly
df = (
    df
    .withColumn("label_4h",  F.when(in_purge_zone, F.lit(None).cast("int")).otherwise(F.col("label_4h")))
    .withColumn("label_24h", F.when(in_purge_zone, F.lit(None).cast("int")).otherwise(F.col("label_24h")))
)

# Assign split column
df = df.withColumn(
    "split",
    F.when(F.col(TS) < F.lit(TEST_START), "train").otherwise("test")
)

# Drop purge-zone rows (null labels)
before_purge = df.count()
df = df.filter(F.col("label_4h").isNotNull() & F.col("label_24h").isNotNull())
after_purge = df.count()

print(f"Purge gap removed {before_purge - after_purge:,} rows ({PURGE_STEPS} steps × {CADENCE_MIN} min)")
print()
df.groupBy("split").count().orderBy("split").show()

In [0]:
# 1. How many unique transformers in each split?
df.groupBy("split", "ID_prefix").count().groupBy("split").count().show()

# 2. Date range per split — confirm test actually spans April to July
df.groupBy("split").agg(
    F.min("DATE").alias("min_date"),
    F.max("DATE").alias("max_date"),
    F.countDistinct("ID_prefix").alias("n_transformers"),
    F.count("*").alias("n_rows")
).show(truncate=False)

# 3. Expected vs actual rows per transformer in test
df.filter(F.col("split") == "test") \
  .groupBy("ID_prefix").count() \
  .agg(F.min("count"), F.max("count"), F.avg("count")) \
  .show()

# 4. How many transformers have data in BOTH splits vs only one?
train_ids = df.filter(F.col("split") == "train").select("ID_prefix").distinct()
test_ids  = df.filter(F.col("split") == "test").select("ID_prefix").distinct()
print("Train-only transformers:", train_ids.subtract(test_ids).count())
print("Test-only transformers:",  test_ids.subtract(train_ids).count())
print("In both splits:",          train_ids.join(test_ids, "ID_prefix").count())

## 9 · Centralised scaling

Fit statistics on **training rows only**, apply to all rows.  
Persist `scaler.json` so inference pipelines can apply the identical transform.

In [0]:
# Columns to z-score (continuous numerics)
ZSCORE_COLS = [
    "current", "voltage",
    "H_LIM_C", "H_LIM_V",
    "load_ratio_c", "load_ratio_v",
    "current_mean_1h", "current_mean_1d", "current_mean_7d",
    "voltage_mean_1h", "voltage_mean_1d", "voltage_mean_7d",
    "current_std_1h",  "current_std_1d",  "current_std_7d",
    "voltage_std_1h",  "voltage_std_1d",  "voltage_std_7d",
    "current_lag_15m", "current_lag_1h",  "current_lag_1d",
    "voltage_lag_15m", "voltage_lag_1h",  "voltage_lag_1d",
    "temp_mean_1d", "temp_mean_7d", "precip_sum_1d",
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m/s",
]

# Columns to log1p then z-score (heavy-tailed / count / max features)
LOG1P_COLS = [
    "current_max_1h", "current_max_1d", "current_max_7d",
    "voltage_max_1h", "voltage_max_1d", "voltage_max_7d",
    "events_15m_cnt",
]

# NOT scaled: is_weekend, hour, day_of_week, month (temporal — model handles encoding)
# NOT scaled: ID_prefix, CONCELHO (categoricals)
# NOT scaled: label_4h, label_24h, split

# Keep only columns that exist in the dataframe
existing = set(df.columns)
ZSCORE_COLS = [c for c in ZSCORE_COLS if c in existing]
LOG1P_COLS  = [c for c in LOG1P_COLS  if c in existing]

print(f"Z-score cols  : {len(ZSCORE_COLS)}")
print(f"Log1p+z cols  : {len(LOG1P_COLS)}")

In [0]:
# ── Fit statistics on TRAINING rows only ───────────────────────────────────────
train_df = df.filter(F.col("split") == "train")

agg_exprs = []
for c in ZSCORE_COLS:
    agg_exprs += [F.avg(c).alias(f"{c}__mu"), F.stddev_pop(c).alias(f"{c}__sd")]
for c in LOG1P_COLS:
    agg_exprs += [
        F.avg(F.log1p(F.col(c))).alias(f"{c}__mu"),
        F.stddev_pop(F.log1p(F.col(c))).alias(f"{c}__sd"),
    ]

stats = train_df.select(*agg_exprs).first().asDict()

def _safe_sd(v):
    """Return std as float; fall back to 1.0 for None or zero (constant column)."""
    return float(v) if (v is not None and float(v) != 0.0) else 1.0

print("Scaler statistics computed on training split ✅")

In [0]:
# ── Apply to all rows ──────────────────────────────────────────────────────────
for c in ZSCORE_COLS:
    mu = float(stats.get(f"{c}__mu") or 0.0)
    sd = _safe_sd(stats.get(f"{c}__sd"))
    df = df.withColumn(c, (F.col(c) - F.lit(mu)) / F.lit(sd))

for c in LOG1P_COLS:
    mu = float(stats.get(f"{c}__mu") or 0.0)
    sd = _safe_sd(stats.get(f"{c}__sd"))
    df = df.withColumn(c, (F.log1p(F.col(c)) - F.lit(mu)) / F.lit(sd))

print("Scaling applied ✅")
print("\nTrain means after scaling (should be ≈ 0):")
display(
    df.filter(F.col("split") == "train")
      .select(*[F.round(F.avg(c), 4).alias(c) for c in ZSCORE_COLS[:6]])
)

In [0]:
# ── Persist scaler.json ────────────────────────────────────────────────────────
scaler_meta = {
    "train_end":       TRAIN_END,
    "test_start":      TEST_START,
    "cadence_min":     CADENCE_MIN,
    "purge_steps":     PURGE_STEPS,
    "zscore_cols":     ZSCORE_COLS,
    "log1p_zscore_cols": LOG1P_COLS,
    "zscore": {
        c: {"mu": float(stats.get(f"{c}__mu") or 0.0), "sd": _safe_sd(stats.get(f"{c}__sd"))}
        for c in ZSCORE_COLS
    },
    "log1p_zscore": {
        c: {"mu": float(stats.get(f"{c}__mu") or 0.0), "sd": _safe_sd(stats.get(f"{c}__sd"))}
        for c in LOG1P_COLS
    },
}

with open("/tmp/scaler.json", "w") as f:
    json.dump(scaler_meta, f, indent=2)

dbutils.fs.mkdirs("dbfs:/gold_features_meta")
dbutils.fs.cp("file:/tmp/scaler.json", SCALER_PATH, recurse=False)
print(f"✅ scaler.json saved to {SCALER_PATH}")

## 10 · Quality checks

In [0]:
print(f"Rows : {df.count():,}")
print(f"Cols : {len(df.columns)}")
print()
df.printSchema()

In [0]:
# ── Label distributions + class imbalance weights per split ────────────────────
# These weights are what each model should use — no resampling needed.
# XGBoost : scale_pos_weight = ratio
# LR      : weightCol (assign weight = ratio to positive rows, 1.0 to negatives)
# LSTM    : pos_weight = torch.tensor([ratio]) in BCEWithLogitsLoss

imbalance_weights = {}   # stored for reference / model notebooks

for label_col in ["label_4h", "label_24h"]:
    print(f"\n{'='*50}")
    print(f"  {label_col}")
    print(f"{'='*50}")

    for split_name in ["train", "test"]:
        split_df = df.filter(F.col("split") == split_name)
        n_total  = split_df.count()
        n_pos    = split_df.filter(F.col(label_col) == 1).count()
        n_neg    = n_total - n_pos
        ratio    = n_neg / builtins.max(n_pos, 1)

        print(f"\n  [{split_name}]")
        print(f"    Total     : {n_total:,}")
        print(f"    Positives : {n_pos:,}  ({100*n_pos/n_total:.2f}%)")
        print(f"    Negatives : {n_neg:,}  ({100*n_neg/n_total:.2f}%)")
        print(f"    neg/pos ratio (imbalance weight) : {ratio:.2f}")

        if split_name == "train":
            imbalance_weights[label_col] = builtins.round(ratio, 4)
            print(f"    → XGBoost  scale_pos_weight      : {ratio:.2f}")
            print(f"    → LSTM     BCEWithLogitsLoss pos_weight : {ratio:.2f}")
            print(f"    → LR       assign weight={ratio:.2f} to label==1 rows")

print(f"\n\nImbalance weights (train split, for model notebooks):")
print(json.dumps(imbalance_weights, indent=2))

In [0]:
# ── Persist dataset_meta.json (imbalance weights for model notebooks) ──────────
# NOTE: imbalance_weights dict is populated in the QA cell (cell-qa-labels).
# Run that cell first, then re-run this one — or move this block to the end
# of cell-qa-labels if you want a single execution order.
dataset_meta = {
    "train_end":          TRAIN_END,
    "test_start":         TEST_START,
    "purge_steps":        PURGE_STEPS,
    "imbalance_weights":  imbalance_weights,   # {"label_4h": X, "label_24h": Y}
}
with open("/tmp/dataset_meta.json", "w") as f:
    json.dump(dataset_meta, f, indent=2)
dbutils.fs.cp("file:/tmp/dataset_meta.json", "dbfs:/gold_features_meta/dataset_meta.json", recurse=False)
print("✅ dataset_meta.json saved to dbfs:/gold_features_meta/dataset_meta.json")

In [0]:
# Null rates — lag/rolling cols will have nulls at the start of each transformer's history
# (expected behaviour — warm-up rows, not a data quality problem)
total = df.count()
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_pct"] = (null_counts["null_count"] / total * 100).round(2)

cols_with_nulls = null_counts[null_counts["null_count"] > 0].sort_values("null_pct", ascending=False)
print(f"Columns with nulls: {len(cols_with_nulls)}")
display(cols_with_nulls)

In [0]:
# Leakage sanity check:
# When the transformer is NOT currently overloaded, label_4h should not be trivially 1.
# If label_4h == 1 for a large fraction of rows where load_ratio <= 1,
# that is expected (it means overload is coming) — not a leakage sign.
# True leakage would show label_4h == 1 = 100% when overloaded AND == 0 otherwise.

check = df.select(
    ((F.col("load_ratio_c") > 0) | (F.col("load_ratio_v") > 0)).alias("currently_overloaded"),
    "label_4h",
    "label_24h"
)

print("=== Label rates by current overload status ===")
print("(load_ratio > 0 after scaling means above-mean loading, not necessarily > limit)")
check.groupBy("currently_overloaded").agg(
    F.round(F.mean("label_4h"),  3).alias("label_4h_rate"),
    F.round(F.mean("label_24h"), 3).alias("label_24h_rate"),
    F.count("*").alias("n_rows")
).orderBy("currently_overloaded").show()

# Confirm no test data bleeds into train
print("=== Date range per split (no overlap expected) ===")
df.groupBy("split").agg(
    F.min(TS).alias("min_date"),
    F.max(TS).alias("max_date")
).orderBy("split").show(truncate=False)

In [0]:
dbutils.data.summarize(df)

## 11 · Save to `gold_features`

Partitioned by `split` so model notebooks can efficiently read just the partition they need:  
```python
spark.read.table("hive_metastore.gold.gold_features").filter(col("split") == "train")
```

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS hive_metastore.gold")

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .partitionBy("split")   # train / test partitions for efficient downstream reads
      .saveAsTable(DST)
)

n_rows = spark.read.table(DST).count()
n_cols = len(spark.read.table(DST).columns)
print(f"\n✅  Saved {DST}")
print(f"   {n_rows:,} rows  |  {n_cols} columns")
print(f"   Partitioned by: split (train / test)")
print(f"   Scaler        : {SCALER_PATH}")
print(f"\n   Labels        : label_4h (4-hour horizon), label_24h (24-hour horizon)")
print(f"   Features      : load ratios, rolling stats (1h/1d/7d), lags (15m/1h/1d),")
print(f"                   weather derived (temp_mean_1d/7d, precip_sum_1d), temporal")
print(f"\n   Imbalance weights (use in model training):")
for k, v in imbalance_weights.items():
    print(f"     {k}: {v}")